# XM655 capture and record

Transmit a tone on the DACs, capture it on all 16 ADCs, look at the IQ.

The capture is **phase aligned**: TX and RX are started by the same trigger, so
every burst begins at the same point in the waveform and the measured phase
means something. In the PL the player enable is `dac_enable OR trig_cap`, so
leaving `dac_enable` low hands the enable to `trig_cap` - one rising edge
restarts the waveform at sample 0 and arms the capture on the same clock.

Run the cells top to bottom.

## 1. Setup

Load the bitstream. This takes a few seconds and only needs to happen once
per kernel.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from lib.mts import doaMtsOverlay
from lib.config_parser import load_config
from lib.common_functions import capture_aligned
from lib.common_functions import convert_raw_to_iq
from lib.common_functions import create_tone_samples
from lib.common_functions import create_zc_chirp_samples
from lib.common_functions import snap_tone_to_fft_bin
from lib.common_functions import write_tone_to_players

CFG = load_config()
overlay = doaMtsOverlay("mts.bit")

## 2. Settings

**Everything you tune lives here.** The cells below only read these.

A few things worth knowing:

- The **rates** are baked into the bitstream, so leave them alone. They are not
  the converter clock rates: the DAC interpolates x10 and the ADC decimates x10.
- The **NCO** sets the RF frequency. `DAC_NCO` is per *tile* (4 numbers), phase
  and gain are per *DAC* (16 numbers) - so all 4 DACs in a tile send the same
  waveform, just shifted and scaled.
- The ADC samples at 2.5 GSPS, so a 4850 MHz signal folds down to
  `5000 - 4850 = 150 MHz`. Rule of thumb: **`ADC_NCO = 5000 - DAC_NCO`**, and the
  tone comes back at its own frequency.
- Don't go much above 4900 MHz. The receive mixer also makes an image
  `2 x fold` away, and if the fold is small the image lands inside the band and
  ruins the signal.
- The baseband **signal itself** (what waveform, `"tone"` or `"chirp"`) is a
  separate block below, and gets built in section 3.

In [ ]:
# --- rates and board size: fixed by the bitstream, from config/parameters.json ---
DAC_SR  = CFG["board"]["dac_sr"]     # DAC baseband rate = 10 GSPS / 10 (C2R eats one x2)
ADC_SR  = CFG["board"]["adc_sr"]     # ADC baseband rate = 2.5 GSPS / 10 decimation
N_CH    = CFG["board"]["n_ch"]       # RF channels on the XM655
N_TILES = CFG["board"]["n_tile"]     # Number of supported tiles (ADC and DAC groupes)

# --- transmit ---
DAC_NCO   = CFG["rf"]["dac_nco"]     # MHz, per tile -> TX lands at 4900
DAC_ZONE  = CFG["rf"]["dac_zone"]    # Nyquist zone, per tile
DAC_PHASE = [0] * N_CH               # degrees, per DAC
DAC_GAIN  = [1] * N_CH               # 0..1, per DAC

# --- capture size ---
N_CAP   = CFG["capture"]["n_cap"]    # samples per channel in one ADC capture

# trig_cap has to stay high for a whole capture window - the DAC only plays
# while the enable is high, so releasing early leaves the tail silent
TRIG_HOLD_S = CFG["capture"]["trig_hold_s"]

# --- receive ---
ADC_NCO   = CFG["rf"]["adc_nco"]     # MHz, per tile = 5000 - DAC_NCO
ADC_ZONE  = CFG["rf"]["adc_zone"]    # fold is in an even zone -> 2
ADC_PHASE = [0] * N_CH               # degrees, per ADC

# --- capture proccessing ---
PLOT_SPECTRUM = True
PLOT_TIME     = True

### Baseband signal configuration

What actually gets played, independent of the RF settings above.

- **`SIG_TYPE = "tone"`** - a single harmonic at `TONE_MHZ`, offset down from
  the tile NCO. The frequency gets snapped in section 3 so it fits a whole
  number of cycles in both the DAC loop and the ADC capture window - otherwise
  the waveform jumps every wrap, or it lands between FFT bins and the measured
  phase is meaningless.
- **`SIG_TYPE = "chirp"`** - a narrowband Zadoff-Chu-style CAZAC chirp: constant
  amplitude, quadratic phase, instantaneous frequency sweeping linearly from 0
  to `ZC_BW_MHZ` across the capture window.

In [ ]:
# --- baseband signal ---
SIG_TYPE  = "tone"                   # "tone" or "chirp"
TONE_MHZ  = CFG["signal"]["tone_mhz"]  # baseband tone, offset down from the NCO
AMP       = CFG["signal"]["amp"]       # 14 bit DAC: +16383 / -16384
ZC_BW_MHZ = 1.0                      # chirp only: instantaneous freq sweeps 0 -> ZC_BW_MHZ

## 3. Generate signal

Build the baseband waveform for `SIG_TYPE`. This is what gets copied into the
DAC player memories in section 4.

In [ ]:
n_samples = overlay.dac0_player.shape[0] // 2

if SIG_TYPE == "tone":
    # the tone must fit a whole number of cycles in BOTH the DAC loop (else the
    # waveform jumps every wrap) and the ADC capture (else it lands between FFT
    # bins and the measured phase is meaningless). ADC_SR / N_CAP is the coarser
    # step and is an exact multiple of the DAC one, so matching it satisfies both.
    TONE_MHZ = snap_tone_to_fft_bin(TONE_MHZ, ADC_SR, N_CAP)
    TONE_HZ = TONE_MHZ * 1e6
    baseband = create_tone_samples(n_samples, DAC_SR, TONE_HZ, AMP)
elif SIG_TYPE == "chirp":
    baseband = create_zc_chirp_samples(n_samples, DAC_SR, ZC_BW_MHZ * 1e6, AMP)
else:
    raise ValueError("SIG_TYPE must be 'tone' or 'chirp', got %r" % SIG_TYPE)

## 4. Transmit

Push the settings into the DACs, copy the signal built in section 3 into all
four tile memories, then **arm without running**: `dacs_off()` leaves
`dac_enable` low so `trig_cap` owns the player enable. Nothing comes out until
section 5 fires the trigger.

In [ ]:
# --- tune ---
overlay.d_centre_freq  = DAC_NCO
overlay.d_nyquist_zone = DAC_ZONE
overlay.d_phases       = DAC_PHASE
overlay.d_gain         = DAC_GAIN
overlay.configure_dacs()

# --- load it into all four tile memories, then arm without running ---
write_tone_to_players(overlay, baseband)

overlay.dacs_off()
print("DACs armed (%s) at %.3f MHz - idle until triggered"
      % (SIG_TYPE, DAC_NCO[0] - (TONE_HZ / 1e6 if SIG_TYPE == "tone" else 0)))

### Optional: steer the beam

Same waveform out of every DAC, phase and gain per DAC. Rerun section 5 after
changing these - the DACs stay armed, so the alignment is not affected.

Reconfiguring between bursts is safe: `configure_dacs()` does call
`ResetNCOPhase()`, but `EVENT_SRC` is `EVNT_SRC_SYSREF`, so that reset lands on
a SYSREF edge and the DAC-to-ADC phase relationship survives it.

In [ ]:
DAC_PHASE = [0, 0, 0, 0] * N_TILES  # degrees, one per DAC (180 exactly is rejected)
DAC_GAIN  = [1, 1, 1, 1] * N_TILES     # 0..1, one per DAC

overlay.d_phases = DAC_PHASE
overlay.d_gain   = DAC_GAIN
overlay.configure_dacs()

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 5. Receive

Each of the 16 ADCs has its own antenna input. Set the channel list and
`data_size` before capturing, otherwise the driver quietly returns only the
first 4 channels, and tune before `configure_adcs()` or the settings land one
run late.

`capture_aligned()` fires the one edge that starts TX and RX together, holds it
for a whole capture window, then reads the buffers with
`_get_selected_xm655_data()`. It deliberately avoids `get_custom_data_xm655()`,
which would fire its own trigger and undo the alignment.

In [ ]:
# --- ask for all 16 channels ---
overlay.active_rf_channels = list(range(N_CH))
overlay.channels  = N_CH * 2                    # I and Q per channel
overlay.data_size = N_CAP * overlay.channels

# --- tune, then configure ---
overlay.centre_freq  = ADC_NCO
overlay.nyquist_zone = ADC_ZONE
overlay.phases       = ADC_PHASE
overlay.configure_adcs()

# --- capture ---
raw = capture_aligned(overlay, TRIG_HOLD_S)
print("raw samples:", raw.size)

## 6. Convert to IQ and plot

`raw` is one long int16 stream, interleaved I/Q per channel. Split it into a
complex array and look at it.

In [ ]:
def plot_iq(iq, max_samples=1000):
    """4x4 grid, I and Q per channel."""
    n = min(max_samples, iq.shape[1])
    fig, axes = plt.subplots(4, 4, figsize=(16, 10), sharex=True)
    for ch, ax in enumerate(axes.flat):
        ax.plot(iq[ch, :n].real, label="I")
        ax.plot(iq[ch, :n].imag, label="Q")
        ax.set_title(f"Channel {ch}")
        ax.grid(True)
    axes.flat[0].legend()
    fig.supxlabel("Sample")
    fig.supylabel("ADC value")
    plt.tight_layout()
    plt.show()


def plot_spectrum(iq):
    """4x4 grid, spectrum per channel. All dB are relative to the strongest
    channel, so a quiet channel looks quiet."""
    n = iq.shape[1]
    f = np.fft.fftshift(np.fft.fftfreq(n, 1 / ADC_SR)) / 1e6
    win = np.hanning(n)

    S = np.abs(np.fft.fftshift(np.fft.fft(iq * win, axis=1), axes=1))
    S = 20 * np.log10(S / S.max() + 1e-12)

    fig, axes = plt.subplots(4, 4, figsize=(16, 10), sharex=True, sharey=True)
    for ch, ax in enumerate(axes.flat):
        ax.plot(f, S[ch])
        ax.set_title(f"Ch {ch}   peak {f[S[ch].argmax()]:+.2f} MHz", fontsize=9)
        ax.grid(True)
    axes.flat[0].set_ylim(-80, 5)
    fig.supxlabel("MHz")
    fig.supylabel("dB (relative to strongest channel)")
    plt.tight_layout()
    plt.show()


iq = convert_raw_to_iq(raw, N_CH)
print(iq.shape)

if PLOT_SPECTRUM:
    plot_spectrum(iq)
if PLOT_TIME:
    plot_iq(iq)

## 7. Stop

Switch the transmitter off when you are done.

In [9]:
overlay.dacs_off()